In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/tips-hindawi/Tips Hindawi University Info.pdf


In [2]:
!pip install sentence-transformers PyPDF2 faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.8 MB/s eta 0:00:00:00:0100:01


In [3]:
# Import
import torch
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd

2026-01-30 13:54:45.681178: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769781285.861356      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769781285.913910      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769781286.348275      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769781286.348314      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769781286.348317      55 computation_placer.cc:177] computation placer alr

In [4]:
# Load Model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [16]:
def generate_text(prompt, max_length=150, return_sequences = 1):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_length,
        num_return_sequences = return_sequences, 
        do_sample=True,
        top_k=50,
        top_p=0.99,
        temperature = 0.001,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

In [6]:
def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

In [7]:
def chunk_text(text, chunk_size=300, overlap = 50):
     words = text.split()
     chunks = []
     for i in range(0, len(words), chunk_size - overlap):
         chunk = " ".join(words[i:i+chunk_size])
         chunks.append(chunk)
     return chunks

In [8]:
def embed_chunks(chunks, model_name = "sentence-transformers/all-MiniLM-L6-v2"):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy = True)
    return model, embeddings

In [9]:
def create_faiss_index(embeddings):
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

In [10]:
def search_index(query, model, index, chuncks, top_k = 3):
    query_embedding = model.encode([query], convert_to_numpy = True)
    distances, indices = index.search(query_embedding, top_k)
    return [chuncks[i] for i in indices[0]]

In [17]:
pdf_path = "/kaggle/input/tips-hindawi/Tips Hindawi University Info.pdf"
text = extract_text(pdf_path)
chunks = chunk_text(text, chunk_size=100, overlap = 10)
model_embeddings, embeddings = embed_chunks(chunks)
index = create_faiss_index(embeddings)

In [45]:
def answer_question_rag(question, top_k=3, max_length=150, verbose=True):
    # 1) Retrieve top-k chunks
    top_matched_chunks = search_index(question, model_embeddings, index, chunks, top_k=top_k)

    # Optional: print retrieved chunks 
    if verbose:
        print("List of Matched Chunks:\n")
        for i, chunk in enumerate(top_matched_chunks):
            print(f"\n--- Chunk {i} ---\n{chunk}\n")

        print("---------------Prompt & Answer---------------\n")

    # 2) Use the best chunk 
    chosen_chunk = top_matched_chunks[0]

    # 3) Build the successful prompt
    prompt = f"""
    Answer the question using ONLY the information in the text below.
    - You may restate or logically infer information that is directly supported by the text
     (Example: if the text mentions scholarships or grants, you may conclude that financial aid exists.)

    Question:
    {question}

    Text:
    {chosen_chunk}

    Answer:
    """

    # 4) Generate answer
    answer = generate_text(prompt, max_length=max_length)

    # 5) Return only the text answer
    return answer[0]


In [46]:
print(answer_question_rag("Where is Tips Hindawi University located?"))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


List of Matched Chunks:


--- Chunk 0 ---
1. General Overview Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle East. Founded in 1963, the university has grown into a globally recognized center for academic excellence and innovation. With over six decades of educational leadership, THU has produced more than 150,000 graduates who serve in diverse industries and academic circles worldwide. The university is accredited by the International Commission for Academic Standards and the Ministry of Higher Education. It operates under the guiding motto: "Knowledge, Integrity, Progress" , and fosters an environment of research, creativity, and public service. 2.


--- Chunk 1 ---
fosters an environment of research, creativity, and public service. 2. Campus and Facilities 2.1 Main Campus The main campus is located in the capital city and spans over 300 acres. It houses: - 12 academic buildings - Central library with over 1 million volume

In [47]:
print(answer_question_rag("Does the university offer online programs?"))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


List of Matched Chunks:


--- Chunk 0 ---
1. General Overview Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle East. Founded in 1963, the university has grown into a globally recognized center for academic excellence and innovation. With over six decades of educational leadership, THU has produced more than 150,000 graduates who serve in diverse industries and academic circles worldwide. The university is accredited by the International Commission for Academic Standards and the Ministry of Higher Education. It operates under the guiding motto: "Knowledge, Integrity, Progress" , and fosters an environment of research, creativity, and public service. 2.


--- Chunk 1 ---
Graduate Programs MBA (Master of Business Administration) MSc in Data Science MA in Psychology PhD in Mechanical Engineering PhD in Political Science 4. Admissions and Tuition 4.1 Undergraduate Admissions High school GPA of 85% or equivalent English proficiency

In [48]:
print(answer_question_rag("Is there financial aid for international students?"))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


List of Matched Chunks:


--- Chunk 0 ---
Graduate Programs MBA (Master of Business Administration) MSc in Data Science MA in Psychology PhD in Mechanical Engineering PhD in Political Science 4. Admissions and Tuition 4.1 Undergraduate Admissions High school GPA of 85% or equivalent English proficiency test (IELTS 6.0 or TOEFL 80) Entrance interview for certain programs 4.2 Graduate Admissions Relevant bachelor’s degree Minimum GPA of 3.0/4.0 Two academic references Statement of purpose 4.3 Tuition Fees (Annual) Undergraduate: $5,000 - $8,000 Graduate: $6,500 - $10,000 4.4 Scholarships Merit Scholarships : 25% to 100% tuition coverage Need-Based Grants Research Fellowships for graduate students 5. Administration President : Dr .


--- Chunk 1 ---
Dormitory (First-year undergraduates) Rafidain Apartments (Graduate students) Amal Housing Complex (Family and international students) 3. Academic Structure 3.1 Faculties Faculty of Engineering Faculty of Medicine and Health Sciences Faculty o

In [49]:
print(answer_question_rag("What languages are used for instruction?"))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


List of Matched Chunks:


--- Chunk 0 ---
Annual "Future Frontiers" Conference Research budget: $12 million annually Sample Project: - "Solar-Powered Desalination Systems for Arid Regions" led by Prof. Amina Al-Rawi 7. Student Life 7.1 Clubs and Organizations THU Debate Society Robotics Club Arabic Calligraphy Circle Environmental Awareness Network International Students Union 7.2 Events and Traditions Founders' Week Annual Cultural Festival Spring Research Showcase 7.3 Support Services Mental Health Counseling Career Development Center Tutoring and Writing Labs Disability Services 8. Faculty Highlights Dr. Yasir Al-Sabbagh (Computer Science): Expert in natural language processing Prof. Rana Khalil (Economics): World Bank consultant and author Dr. Omar Al-Mansoori (Mechanical Engineering): Holder of


--- Chunk 1 ---
Dormitory (First-year undergraduates) Rafidain Apartments (Graduate students) Amal Housing Complex (Family and international students) 3. Academic Structure 3.1 Faculties 